# Importing Libraries

In [1]:
import pandas as pd

# Loading and reading the file

In [2]:
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

# Data Cleaning

In [3]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Checking Missing Values

In [4]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


# Creating Tenure Group
How long has the customer been with the company (in months)

In [5]:
def tenure_group(tenure):
    if tenure <= 12:
        return "0-12 months"
    elif tenure <= 24:
        return "12-24 months"
    elif tenure <= 48:
        return "24-48 months"
    elif tenure <= 60:
        return "48-60 months"
    else:
        return "60+ months"

df["TenureGroup"] = df["tenure"].apply(tenure_group)

In [6]:
print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... TechSupport  \
0  No phone service             DSL             No  ...          No   
1                No             DSL            Yes  ...          No   
2                No             DSL            Yes  ...          No   
3  No phone service             DSL            Yes  ...         Yes   
4                No     Fiber optic             No  ...          No   

  StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No             

# Creating a Monthly charges group
Dividing the customer's monthly charges into catogories so it will be easy to analyse

In [7]:
df["MonthlyChargesGroup"] = pd.cut(
    df["MonthlyCharges"],
    bins=[0, 35, 70, 100, 150],
    labels=["Low", "Medium", "High", "Very High"]
)

In [8]:
print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... StreamingTV  \
0  No phone service             DSL             No  ...          No   
1                No             DSL            Yes  ...          No   
2                No             DSL            Yes  ...          No   
3  No phone service             DSL            Yes  ...          No   
4                No     Fiber optic             No  ...          No   

  StreamingMovies        Contract PaperlessBilling              PaymentMethod  \
0          

# Customer Lifetime Value

In [9]:
df["CLV"] = df["MonthlyCharges"] * df["tenure"]

In [10]:
print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... StreamingMovies  \
0  No phone service             DSL             No  ...              No   
1                No             DSL            Yes  ...              No   
2                No             DSL            Yes  ...              No   
3  No phone service             DSL            Yes  ...              No   
4                No     Fiber optic             No  ...              No   

         Contract PaperlessBilling              PaymentMethod Monthl

# Churn Analysis


**Overall Churn Rate**

In [11]:
churn_rate = df['Churn'].mean()*100
print(f"Overall Churn Rate: {churn_rate:.2f}%")

Overall Churn Rate: 26.54%


**Churn Count**


*  0 → customers who stayed
*  1 → customers who left



In [12]:
df['Churn'].value_counts()

,count
Churn,
0,5174
1,1869


# Churn Rate by Contract Type

In [13]:
df.groupby("Contract")["Churn"].mean() * 100

,Churn
Contract,
Month-to-month,42.709677
One year,11.269518
Two year,2.831858


# Churn by Tenure Group

In [14]:
df.groupby("TenureGroup")["Churn"].mean() * 100

,Churn
TenureGroup,
0-12 months,47.438243
12-24 months,28.710938
24-48 months,20.388959
48-60 months,14.423077
60+ months,6.609808


# Churn by Monthly Charges

In [15]:
df.groupby("MonthlyChargesGroup")["Churn"].mean() * 100

/tmp/ipykernel_854/4143550158.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("MonthlyChargesGroup")["Churn"].mean() * 100


,Churn
MonthlyChargesGroup,
Low,10.893372
Medium,23.942029
High,37.821708
Very High,28.048780


# Comparing Average Tenure of Churned vs Retained
* Retained (0)
* Churned (1)

In [16]:
df.groupby("Churn")["tenure"].mean()

,tenure
Churn,
0,37.569965
1,17.979133


# Exporting Clean Data

In [17]:
df.to_csv("cleaned_churn_data.csv", index=False)